Reading existing table



In [0]:
raw_df = spark.table(
    "workspace.default.ps_20174392719_1491204439457_log"
    )
display(raw_df.limit(10))

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
161,PAYMENT,18003.02,C414361072,0.0,0.0,M1783921537,0.0,0.0,0,0
161,PAYMENT,7175.57,C931295611,0.0,0.0,M2007760563,0.0,0.0,0,0
161,PAYMENT,18672.42,C2050705555,0.0,0.0,M863681574,0.0,0.0,0,0
161,PAYMENT,14636.07,C391580315,0.0,0.0,M1059135652,0.0,0.0,0,0
161,PAYMENT,24088.99,C536778755,0.0,0.0,M2030275670,0.0,0.0,0,0
161,PAYMENT,25440.18,C1445941823,0.0,0.0,M1623030878,0.0,0.0,0,0
161,PAYMENT,1530.45,C191422154,0.0,0.0,M1852605133,0.0,0.0,0,0
161,PAYMENT,8062.29,C1123982229,0.0,0.0,M1235894296,0.0,0.0,0,0
161,PAYMENT,1702.61,C75061609,0.0,0.0,M235522497,0.0,0.0,0,0
161,PAYMENT,8862.39,C949868966,0.0,0.0,M590144557,0.0,0.0,0,0


In [0]:
raw_df.count()

6362620

Adding Bronze Metadata

In [0]:
from pyspark.sql.functions import current_timestamp,lit
bronze_df = (
    raw_df
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
            )
    .withColumn(
        "source_system",
        lit("paysim")
    )
)

Creating Bronze Table

In [0]:
(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.bronze_transactions"
        )
)
    

Validation

In [0]:
spark.sql("""
          SELECT count(*)
          FROM workspace.default.bronze_transactions
          """).show()

+--------+
|count(*)|
+--------+
| 6362620|
+--------+



In [0]:
spark.sql("""
          SELECT *
          FROM workspace.default.bronze_transactions
          """).limit(5).show()
          

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+--------------------+-------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud| ingestion_timestamp|source_system|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+--------------------+-------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|2026-06-22 22:02:...|       paysim|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|2026-06-22 22:02:...|       paysim|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|2026-06-22 22:02:...|       paysim|
|   